In [21]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
df = pd.read_csv("../data/processed/cleaned_reviews.csv")

In [23]:
df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,Empty,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,Empty,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
3,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
4,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses


In [24]:
df.shape

(19817, 10)

In [25]:
df.columns

Index(['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='str')

In [26]:
df.dtypes

Clothing ID                int64
Age                        int64
Title                        str
Review Text                  str
Rating                     int64
Recommended IND            int64
Positive Feedback Count    int64
Division Name                str
Department Name              str
Class Name                   str
dtype: object

In [27]:
df.isnull().sum()

Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64

In [28]:
df.duplicated().sum()

np.int64(0)

In [29]:
X = df["Review Text"]
y = df["Recommended IND"]
X.head()
y.head()

0    1
1    1
2    1
3    1
4    0
Name: Recommended IND, dtype: int64

In [30]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [31]:
stop_words = set(stopwords.words("english"))
stop_words.discard("not")
stop_words.discard("no")
stop_words.discard("nor")

print(len(stop_words))

195


In [32]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

In [33]:
x_clean = X.apply(clean_text)

x_clean.head()

0          absolutely wonderful silky sexy comfortable
1    love dress sooo pretty happened find store im ...
2    love love love jumpsuit fun flirty fabulous ev...
3    shirt flattering due adjustable front tie perf...
4    love tracy reese dresses one petite feet tall ...
Name: Review Text, dtype: str

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    x_clean,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (15853,)
X_test: (3964,)
y_train: (15853,)
y_test: (3964,)


In [35]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train TF-IDF shape:", X_train_tfidf.shape)
print("X_test TF-IDF shape:", X_test_tfidf.shape)

X_train TF-IDF shape: (15853, 5000)
X_test TF-IDF shape: (3964, 5000)


In [36]:
import joblib

joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("TF-IDF Vectorizer saved successfully!")

TF-IDF Vectorizer saved successfully!


In [37]:
loaded_tfidf = joblib.load("tfidf_vectorizer.pkl")

print("Vectorizer loaded successfully!")
print("Number of features:", len(loaded_tfidf.get_feature_names_out()))

Vectorizer loaded successfully!
Number of features: 5000


In [38]:
from scipy.sparse import save_npz

save_npz("X_train_tfidf.npz", X_train_tfidf)
save_npz("X_test_tfidf.npz", X_test_tfidf)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Train/Test data saved successfully!")

Train/Test data saved successfully!


In [39]:
import os

files = [
    "tfidf_vectorizer.pkl",
    "X_train_tfidf.npz",
    "X_test_tfidf.npz",
    "y_train.csv",
    "y_test.csv"
]

for file in files:
    print(file, "→", "Exists" if os.path.exists(file) else "Missing")

tfidf_vectorizer.pkl → Exists
X_train_tfidf.npz → Exists
X_test_tfidf.npz → Exists
y_train.csv → Exists
y_test.csv → Exists


In [40]:

print("Original samples:", len(X))
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("TF-IDF max features:", 5000)
print("X_train TF-IDF shape:", X_train_tfidf.shape)
print("X_test TF-IDF shape:", X_test_tfidf.shape)
print("Random state:", 42)

Original samples: 19817
Training samples: 15853
Testing samples: 3964
TF-IDF max features: 5000
X_train TF-IDF shape: (15853, 5000)
X_test TF-IDF shape: (3964, 5000)
Random state: 42
